In [1]:
# =========================================================
# EVALUATION SCRIPT - TEMPO DATASET (PER-SCENARIO GOLD + ALL RATIO)
# =========================================================

import re
import sys
import subprocess
import pandas as pd
import numpy as np

packages = {
    "openpyxl"   : "openpyxl",
    "bert_score" : "bert_score",
    "nltk"       : "nltk",
    "rouge_score": "rouge_score",
}

for pip_name, import_name in packages.items():
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", pip_name, "--break-system-packages", "--quiet"],
            check=True,
        )

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

EXCEL_FILE = "Validasi Ringkasan Sistem (Tugas Akhir)_Andika Prasetya (1).xlsx"
OUT_FILE   = "evaluation_results_tempo_all_ratio_rapi.xlsx"

SCENARIOS = {
    "Premise_Highlight_Tempo": {
        "method"     : "Highlight",
        "gold_file"  : "silver_summary_highlight_tempo.jsonl",
        "sheet_name" : "Highlight Tempo",
        "pred_cols"  : ["Summary 25%", "Summary 50%", "Summary 75%", "Summary 100%"],
    },
    "Premise_First_Sentence_Tempo": {
        "method"     : "First Sentence",
        "gold_file"  : "silver_sum_fixed.jsonl",
        "sheet_name" : "First Sentences Tempo",
        "pred_cols"  : ["Summary 25%", "Summary 50%", "Summary 75%", "Summary 100%"],
    },
    "Premise_Last_Sentence_Tempo": {
        "method"     : "Last Sentence",
        "gold_file"  : "silver_summary_last_tempo.jsonl",
        "sheet_name" : "Last Sentences Tempo",
        "pred_cols"  : ["Summary 25%", "Summary 50%", "Summary 75%", "Summary 100%"],
    },
}

GOLD_SENTENCE_LIMIT = 1
ROUGE_MODE = "recall"

METHOD_ORDER = ["Highlight", "First Sentence", "Last Sentence"]
RATIO_ORDER  = ["25%", "50%", "75%", "100%"]

def flatten_summary(value) -> str:
    if isinstance(value, list):
        return " ".join(str(x) for x in value if x)
    if isinstance(value, str):
        return value.strip()
    return str(value).strip()

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r"(\d),(\d)", r"\1.\2", text)
    text = re.sub(r"[^\w\s\.]", " ", text)
    text = " ".join(text.split())
    return text

def truncate_gold(summary: str, max_sent: int = 1) -> str:
    if not isinstance(summary, str):
        return ""
    summary = summary.strip()
    if not summary:
        return ""
    sents = re.split(r'(?<=[.!?])\s+', summary)
    return " ".join(sents[:max_sent]).strip()

def load_gold(gold_file: str) -> pd.DataFrame:
    df = pd.read_json(gold_file, lines=True)

    for col in ["doc_id", "silver_summary"]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ditemukan di {gold_file}!")

    df["gold_summary_full"] = df["silver_summary"].apply(flatten_summary)
    df["gold_summary"] = df["gold_summary_full"].apply(
        lambda x: truncate_gold(x, max_sent=GOLD_SENTENCE_LIMIT)
    )

    df["doc_id"] = df["doc_id"].astype(str)
    df["index"] = df["doc_id"].str.extract(r"(\d+)")[0].pipe(pd.to_numeric, errors="coerce")

    df = (
        df.dropna(subset=["index"])
          .assign(index=lambda d: d["index"].astype(int))
          .sort_values("index")
          .reset_index(drop=True)
    )
    return df

def compute_rouge(pred: str, ref: str, scorer, mode: str) -> dict:
    pred = normalize_text(pred)
    ref  = normalize_text(ref)

    if not pred or not ref:
        return {"rouge1": None, "rouge2": None, "rougeL": None}

    scores = scorer.score(ref, pred)

    if mode == "recall":
        return {
            "rouge1": scores["rouge1"].recall,
            "rouge2": scores["rouge2"].recall,
            "rougeL": scores["rougeL"].recall,
        }

    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }

def compute_bleu(pred: str, ref: str, smooth):
    pred = normalize_text(pred)
    ref  = normalize_text(ref)

    if not pred or not ref:
        return None

    return sentence_bleu(
        [ref.split()],
        pred.split(),
        weights=(0.5, 0.5),
        smoothing_function=smooth,
    )

rouge_sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
smooth = SmoothingFunction().method1

all_eval_dfs = {}
summary_rows = []

for scenario_name, cfg in SCENARIOS.items():
    method_name = cfg["method"]

    print()
    print("=" * 60)
    print(f"SCENARIO: {scenario_name}")
    print(f"METHOD  : {method_name}")
    print("=" * 60)

    df_gold = load_gold(cfg["gold_file"])
    print(f"  Gold file: {cfg['gold_file']} ({len(df_gold)} dok)")

    try:
        df_pred_raw = pd.read_excel(EXCEL_FILE, sheet_name=cfg["sheet_name"])
    except Exception as e:
        print(f"  [ERROR] Gagal baca sheet '{cfg['sheet_name']}': {e}")
        continue

    for pred_col in cfg["pred_cols"]:
        ratio_clean = pred_col.replace("Summary ", "").strip()
        ratio_name  = ratio_clean.replace("%", "pct")
        scenario_ratio_name = f"{method_name}_{ratio_name}".replace(" ", "_")

        print()
        print("-" * 60)
        print(f"RASIO: {ratio_clean}")
        print("-" * 60)

        if "No" not in df_pred_raw.columns or pred_col not in df_pred_raw.columns:
            print(f"  [SKIP] Kolom 'No' atau '{pred_col}' tidak ditemukan di sheet.")
            continue

        df_pred = df_pred_raw[["No", pred_col]].rename(
            columns={"No": "index", pred_col: "pred"}
        )

        df_pred["index"] = pd.to_numeric(df_pred["index"], errors="coerce")
        df_pred = (
            df_pred.dropna(subset=["index"])
                   .assign(index=lambda d: d["index"].astype(int))
        )

        df = df_gold[["index", "doc_id", "gold_summary", "gold_summary_full"]].merge(
            df_pred,
            on="index",
            how="left"
        )

        print(f"  Prediksi tersedia: {df['pred'].notna().sum()}")

        results = []

        for _, row in df.iterrows():
            gold = row["gold_summary"]

            if not isinstance(gold, str) or not gold.strip():
                continue

            pred = row.get("pred", "")
            if not isinstance(pred, str):
                pred = ""

            rouge_r = compute_rouge(pred, gold, rouge_sc, ROUGE_MODE)
            bleu_r  = compute_bleu(pred, gold, smooth)

            results.append({
                "doc_id"            : row["doc_id"],
                "index"             : row["index"],
                "metode"            : method_name,
                "rasio"             : ratio_clean,
                "gold_words"        : len(gold.split()),
                "gold_summary"      : gold,
                "gold_summary_full" : row["gold_summary_full"],
                "pred_words"        : len(pred.split()) if pred else 0,
                "rouge1"            : rouge_r["rouge1"],
                "rouge2"            : rouge_r["rouge2"],
                "rougeL"            : rouge_r["rougeL"],
                "bleu"              : bleu_r,
                "pred"              : pred,
            })

        eval_df = pd.DataFrame(results)
        print(f"  Total dievaluasi: {len(eval_df)} dokumen")

        valid_pred, valid_gold, valid_idx = [], [], []

        for i, row in eval_df.iterrows():
            if str(row["pred"]).strip() and str(row["gold_summary"]).strip():
                valid_pred.append(str(row["pred"]).strip())
                valid_gold.append(str(row["gold_summary"]).strip())
                valid_idx.append(i)

        bs_values = [None] * len(eval_df)

        if valid_pred:
            print(f"  Menjalankan BERTScore untuk {len(valid_pred)} dokumen...")
            P, R, F1 = bert_score(
                valid_pred,
                valid_gold,
                model_type="xlm-roberta-base",
                lang="id",
                verbose=False,
            )

            for j, idx in enumerate(valid_idx):
                bs_values[idx] = F1[j].item()

        eval_df["bertscore"] = bs_values

        r1 = eval_df["rouge1"].dropna().mean()
        r2 = eval_df["rouge2"].dropna().mean()
        rl = eval_df["rougeL"].dropna().mean()
        bl = eval_df["bleu"].dropna().mean()
        bs = eval_df["bertscore"].dropna().mean()

        summary_rows.append({
            "Metode"       : method_name,
            "Rasio"        : ratio_clean,
            "Matched"      : int((eval_df["pred_words"] > 0).sum()),
            "ROUGE-1"      : round(r1, 4),
            "ROUGE-2"      : round(r2, 4),
            "ROUGE-L"      : round(rl, 4),
            "BLEU"         : round(bl, 4),
            "BERTScore-F1" : round(bs, 4),
            "BERT-Matched" : int(eval_df["bertscore"].notna().sum()),
        })

        all_eval_dfs[scenario_ratio_name] = eval_df

summary_df = pd.DataFrame(summary_rows)

if not summary_df.empty:
    summary_df["Metode"] = pd.Categorical(summary_df["Metode"], categories=METHOD_ORDER, ordered=True)
    summary_df["Rasio"] = pd.Categorical(summary_df["Rasio"], categories=RATIO_ORDER, ordered=True)
    summary_df = summary_df.sort_values(["Metode", "Rasio"]).reset_index(drop=True)

print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

detail_cols = [
    "doc_id", "index", "metode", "rasio",
    "gold_words", "gold_summary", "gold_summary_full",
    "pred_words", "rouge1", "rouge2", "rougeL", "bleu", "bertscore",
    "pred",
]

with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

    for sheet_name, eval_df in all_eval_dfs.items():
        eval_df[detail_cols].to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"✅ Hasil disimpan ke: {OUT_FILE}")

d:\conda_envs\nlp_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\conda_envs\nlp_project\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(



SCENARIO: Premise_Highlight_Tempo
METHOD  : Highlight
  Gold file: silver_summary_highlight_tempo.jsonl (100 dok)

------------------------------------------------------------
RASIO: 25%
------------------------------------------------------------
  Prediksi tersedia: 100
  Total dievaluasi: 100 dokumen
  Menjalankan BERTScore untuk 100 dokumen...


d:\conda_envs\nlp_project\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0525 19:13:03.662494 34320 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
d:\conda_envs\nlp_project\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(



------------------------------------------------------------
RASIO: 50%
------------------------------------------------------------
  Prediksi tersedia: 100
  Total dievaluasi: 100 dokumen
  Menjalankan BERTScore untuk 100 dokumen...

------------------------------------------------------------
RASIO: 75%
------------------------------------------------------------
  Prediksi tersedia: 100
  Total dievaluasi: 100 dokumen
  Menjalankan BERTScore untuk 100 dokumen...

------------------------------------------------------------
RASIO: 100%
------------------------------------------------------------
  Prediksi tersedia: 100
  Total dievaluasi: 100 dokumen
  Menjalankan BERTScore untuk 100 dokumen...

SCENARIO: Premise_First_Sentence_Tempo
METHOD  : First Sentence
  Gold file: silver_sum_fixed.jsonl (100 dok)

------------------------------------------------------------
RASIO: 25%
------------------------------------------------------------
  Prediksi tersedia: 100
  Total dievaluasi: 1

In [2]:
# =========================================================
# EVALUATION SCRIPT - TEMPO JSONL vs JSONL
# =========================================================

import re
import sys
import subprocess
import pandas as pd
import numpy as np
from typing import Optional

# =========================================================
# INSTALL PACKAGE
# =========================================================
packages = {
    "openpyxl"   : "openpyxl",
    "bert_score" : "bert_score",
    "nltk"       : "nltk",
    "rouge_score": "rouge_score",
}

for pip_name, import_name in packages.items():
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", pip_name,
             "--break-system-packages", "--quiet"],
            check=True,
        )

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

# =========================================================
# FILE CONFIG
# =========================================================
GOLD_FILE = "silver_sum_fixed.jsonl"
PRED_FILE = "tempo_hybrid_results_textrank_fix.jsonl"
OUT_FILE  = "evaluation_results_textrank_fix.xlsx"

# =========================================================
# CONFIG
# =========================================================
GOLD_SENTENCE_LIMIT = 1
ROUGE_MODE          = "recall"

# =========================================================
# HELPER
# =========================================================
def flatten_summary(value) -> str:
    if isinstance(value, list):
        return " ".join(str(x) for x in value if x)
    if isinstance(value, str):
        return value.strip()
    return str(value).strip()


def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r"(\d),(\d)", r"\1.\2", text)
    text = re.sub(r"[^\w\s\.]", " ", text)
    text = " ".join(text.split())
    return text


def truncate_gold(summary: str, max_sent: int = 1) -> str:
    if not isinstance(summary, str):
        return ""
    summary = summary.strip()
    if not summary:
        return ""
    sents = re.split(r'(?<=[.!?])\s+', summary)
    return " ".join(sents[:max_sent]).strip()


def extract_index(doc_id: str) -> Optional[int]:
    m = re.search(r"(\d+)", str(doc_id))
    return int(m.group(1)) if m else None


def compute_rouge(pred: str, ref: str, scorer, mode: str) -> dict:
    pred = normalize_text(pred)
    ref  = normalize_text(ref)
    if not pred or not ref:
        return {"rouge1": None, "rouge2": None, "rougeL": None}
    scores = scorer.score(ref, pred)
    if mode == "recall":
        return {
            "rouge1": scores["rouge1"].recall,
            "rouge2": scores["rouge2"].recall,
            "rougeL": scores["rougeL"].recall,
        }
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def compute_bleu(pred: str, ref: str, smooth) -> Optional[float]:
    pred = normalize_text(pred)
    ref  = normalize_text(ref)
    if not pred or not ref:
        return None
    return sentence_bleu(
        [ref.split()], pred.split(),
        weights=(0.5, 0.5),
        smoothing_function=smooth,
    )


# =========================================================
# LOAD GOLD
# =========================================================
print("=" * 60)
print("LOAD GOLD")
print("=" * 60)

df_gold = pd.read_json(GOLD_FILE, lines=True)
for col in ["doc_id", "silver_summary"]:
    if col not in df_gold.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan di gold!")

df_gold["gold_summary_full"] = df_gold["silver_summary"].apply(flatten_summary)
df_gold["gold_summary"]      = df_gold["gold_summary_full"].apply(
    lambda x: truncate_gold(x, max_sent=GOLD_SENTENCE_LIMIT)
)
df_gold["doc_id"] = df_gold["doc_id"].astype(str)
df_gold["index"]  = df_gold["doc_id"].apply(extract_index)
df_gold = (
    df_gold.dropna(subset=["index"])
           .assign(index=lambda d: d["index"].astype(int))
           .sort_values("index")
           .reset_index(drop=True)
)
print(f"Total dokumen gold: {len(df_gold)}")

# =========================================================
# LOAD PREDICTION
# =========================================================
print()
print("=" * 60)
print("LOAD PREDICTION")
print("=" * 60)

df_pred = pd.read_json(PRED_FILE, lines=True)
print(f"Kolom tersedia di pred file: {list(df_pred.columns)}")

# Cari kolom index/doc_id di pred file
index_col = None
for candidate in ["doc_id", "id", "index", "No", "no"]:
    if candidate in df_pred.columns:
        index_col = candidate
        break

if index_col is None:
    # Tidak ada kolom id — pakai urutan baris langsung (asumsi sudah urut 1-100)
    print("  Tidak ada kolom id ditemukan — pakai urutan baris (1-based)")
    df_pred["index"] = range(1, len(df_pred) + 1)
else:
    print(f"  Menggunakan kolom '{index_col}' sebagai index")
    df_pred["index"] = df_pred[index_col].apply(
        lambda x: extract_index(str(x)) if not str(x).isdigit() else int(x)
    )
    df_pred = (
        df_pred.dropna(subset=["index"])
               .assign(index=lambda d: d["index"].astype(int))
               .sort_values("index")
               .reset_index(drop=True)
    )

if "Abstractive_Summary" not in df_pred.columns:
    raise ValueError(
        f"Kolom 'Abstractive_Summary' tidak ditemukan! "
        f"Kolom tersedia: {list(df_pred.columns)}"
    )

df_pred = df_pred[["index", "Abstractive_Summary"]].rename(
    columns={"Abstractive_Summary": "pred"}
)
print(f"Total prediksi   : {len(df_pred)}")

# =========================================================
# MERGE
# =========================================================
df = df_gold[["index", "doc_id", "gold_summary", "gold_summary_full"]].merge(
    df_pred, on="index", how="left"
)
n_matched = df["pred"].notna().sum()
print(f"Matched setelah merge: {n_matched} / {len(df)}")

# Cek jika ada yang tidak match
unmatched = df[df["pred"].isna()]["index"].tolist()
if unmatched:
    print(f"  [WARN] Index tidak match: {unmatched[:10]}{'...' if len(unmatched) > 10 else ''}")

# =========================================================
# EVALUATION
# =========================================================
print()
print("=" * 60)
print("EVALUATION")
print("=" * 60)

rouge_sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
smooth   = SmoothingFunction().method1

results = []
for _, row in df.iterrows():
    gold = row["gold_summary"]
    if not isinstance(gold, str) or not gold.strip():
        continue

    pred = row.get("pred", "")
    if not isinstance(pred, str):
        pred = ""

    rouge_r = compute_rouge(pred, gold, rouge_sc, ROUGE_MODE)
    bleu_r  = compute_bleu(pred, gold, smooth)

    results.append({
        "doc_id"           : row["doc_id"],
        "index"            : row["index"],
        "gold_words"       : len(gold.split()),
        "gold_summary"     : gold,
        "gold_summary_full": row["gold_summary_full"],
        "pred_words"       : len(pred.split()) if pred else 0,
        "pred"             : pred,
        "rouge1"           : rouge_r["rouge1"],
        "rouge2"           : rouge_r["rouge2"],
        "rougeL"           : rouge_r["rougeL"],
        "bleu"             : bleu_r,
    })

eval_df = pd.DataFrame(results)
print(f"Total dievaluasi: {len(eval_df)} dokumen")

# =========================================================
# BERTSCORE
# =========================================================
print()
print("=== RUNNING BERTSCORE (xlm-roberta-base) ===")

valid_pred, valid_gold, valid_idx = [], [], []
for i, row in eval_df.iterrows():
    if isinstance(row["pred"], str) and row["pred"].strip() \
       and isinstance(row["gold_summary"], str) and row["gold_summary"].strip():
        valid_pred.append(row["pred"].strip())
        valid_gold.append(row["gold_summary"].strip())
        valid_idx.append(i)

bs_values = [None] * len(eval_df)
if valid_pred:
    print(f"  Menjalankan BERTScore untuk {len(valid_pred)} dokumen...")
    P, R, F1 = bert_score(
        valid_pred, valid_gold,
        model_type="xlm-roberta-base",
        lang="id",
        verbose=False,
    )
    for j, idx in enumerate(valid_idx):
        bs_values[idx] = F1[j].item()

eval_df["bertscore"] = bs_values

# =========================================================
# SUMMARY
# =========================================================
r1 = eval_df["rouge1"].dropna().mean()
r2 = eval_df["rouge2"].dropna().mean()
rl = eval_df["rougeL"].dropna().mean()
bl = eval_df["bleu"].dropna().mean()
bs = eval_df["bertscore"].dropna().mean()

print()
print("=" * 60)
print(f"RESULTS (ROUGE = {ROUGE_MODE.upper()} | Gold = first sentence)")
print("=" * 60)
print(f"  Dokumen dievaluasi : {len(eval_df)}")
print(f"  ROUGE-1            : {r1:.4f}")
print(f"  ROUGE-2            : {r2:.4f}")
print(f"  ROUGE-L            : {rl:.4f}")
print(f"  BLEU               : {bl:.4f}")
print(f"  BERTScore-F1       : {bs:.4f}")

summary_df = pd.DataFrame([{
    "Model"       : "TextRank_IndoT5_Fix",
    "Gold File"   : GOLD_FILE,
    "Pred File"   : PRED_FILE,
    "Matched"     : int((eval_df["pred_words"] > 0).sum()),
    "ROUGE-1"     : round(r1, 4),
    "ROUGE-2"     : round(r2, 4),
    "ROUGE-L"     : round(rl, 4),
    "BLEU"        : round(bl, 4),
    "BERTScore-F1": round(bs, 4),
    "BERT-Matched": int(eval_df["bertscore"].notna().sum()),
}])

# =========================================================
# SAVE EXCEL
# =========================================================
detail_cols = [
    "doc_id", "index",
    "gold_words", "gold_summary", "gold_summary_full",
    "pred_words", "rouge1", "rouge2", "rougeL", "bleu", "bertscore",
    "pred",
]

with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    eval_df[detail_cols].to_excel(writer, sheet_name="Detail", index=False)

print()
print(f"✅ Hasil disimpan ke: {OUT_FILE}")

LOAD GOLD
Total dokumen gold: 100

LOAD PREDICTION
Kolom tersedia di pred file: ['doc_id', 'No', 'Content', 'Extractive_Summary', 'Abstractive_Summary']
  Menggunakan kolom 'doc_id' sebagai index
Total prediksi   : 300
Matched setelah merge: 100 / 100

EVALUATION
Total dievaluasi: 100 dokumen

=== RUNNING BERTSCORE (xlm-roberta-base) ===
  Menjalankan BERTScore untuk 100 dokumen...

RESULTS (ROUGE = RECALL | Gold = first sentence)
  Dokumen dievaluasi : 100
  ROUGE-1            : 0.4122
  ROUGE-2            : 0.2438
  ROUGE-L            : 0.3572
  BLEU               : 0.1966
  BERTScore-F1       : 0.8729

✅ Hasil disimpan ke: evaluation_results_textrank_fix.xlsx
